# 00 — Setup
Run this cell at the top of every notebook (Part 0). Mounts Drive, clones/pulls the repo, sets up persistent artifact dirs, installs pinned deps and the CLI RTL toolchain.

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

REPO = "https://<TOKEN>@github.com/<you>/verilog-slm.git"  # fill in
if not os.path.exists('/content/verilog-slm'):
    !git clone {REPO} /content/verilog-slm
%cd /content/verilog-slm
!git pull

CKPT = '/content/drive/MyDrive/verilog-slm/checkpoints'
LOGS = '/content/drive/MyDrive/verilog-slm/logs'
os.makedirs(CKPT, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
!ln -sfn {CKPT} artifacts_drive_ckpt
!ln -sfn {LOGS} artifacts_drive_logs

In [ ]:
# Pinned deps (Part 0 hygiene: Colab silently upgrades packages between sessions)
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
# RTL toolchain: iverilog is required (compile+simulate). yosys and verible
# are optional -- the harness soft-gates on them (see docs/industry_standards.md)
# but install them here so the synthesis/lint stages actually run instead of
# being recorded as 'skipped'.
!apt-get -qq update && apt-get -qq install -y iverilog yosys > /dev/null
!curl -sL https://github.com/chipsalliance/verible/releases/latest/download/verible-linux-static-x86_64.tar.gz -o /tmp/verible.tar.gz
!mkdir -p /opt/verible && tar -xzf /tmp/verible.tar.gz -C /opt/verible --strip-components=1
os.environ['PATH'] += ':/opt/verible/bin'
!iverilog -V | head -1
!yosys -V
!verible-verilog-lint --version

In [ ]:
# Record the GPU model at the start of every run -- required for the
# per-GPU-hour metric (Part 0 non-negotiable hygiene) to mean anything.
!nvidia-smi -L

In [ ]:
# Smoke test: the pure-Python side of the pipeline (no GPU/tools needed)
!python -m pytest -q tests/ -x